# Widgets & Configuration

In [0]:

# 01 — Data Profiling
# Profiles all five raw VStone landing files: null/completeness audit,
# distinct-value counts, and a primary-key/duplicate check per file.

# `streets.csv` (87.8M rows, street-level sensor readings) was landed
# after the initial profiling pass and added here — see
# `docs/requirements_and_assumptions.md` assumption #11. It is now the
# **chunked** dataset (see `02_data_chunking.py`); `cars.csv` moved to the
# whole-load group alongside `telegram.csv` / `node_locations.csv` /
# `streets_list.csv` — it still gets profiled and still needs full
# Bronze→Silver→Gold treatment per the PDF, it's just no longer the file
# being split into 4 ingestion-technique chunks.

# Adapted from the reference project's `01_data_profiling.py.py` audit
# pattern (same metrics, same style of output table), retargeted at the
# VStone datasets. The Cyrillic-composite-PK branch used for the
# reference's `catalogs.csv` doesn't apply here and was removed rather
# than adapted — VStone has no equivalent composite-key file.
 
import pandas as pd
from pyspark.sql.functions import (
    col, count, when, isnull, trim, countDistinct
)
 
dbutils.widgets.text("catalog_name", "vstone_catalog", "1. Catalog Name")
dbutils.widgets.text("raw_schema", "raw", "2. Raw Schema")
dbutils.widgets.text("landing_volume", "landing", "3. Landing Volume")
 
CATALOG = dbutils.widgets.get("catalog_name")
RAW_SCHEMA = dbutils.widgets.get("raw_schema")
LANDING_VOL = dbutils.widgets.get("landing_volume")
 
LANDING_PATH = f"/Volumes/{CATALOG}/{RAW_SCHEMA}/{LANDING_VOL}"
 
# File manifest — reads every column as STRING (inferSchema=false) for
# string-safe profiling, matching the reference project's convention.
FILES = {
    "cars": {
        "path": f"{LANDING_PATH}/cars.csv",
        "opts": {"header": "true"},
        "pk_candidate": None,  # no single-column key — see assumptions doc
    },
    "telegram": {
        "path": f"{LANDING_PATH}/telegram.csv",
        # multi-line quoted free text — naive comma-split corrupts this file
        "opts": {"header": "true", "multiLine": "true", "escape": '"', "quote": '"'},
        "pk_candidate": None,  # no id column at all
    },
    "node_locations": {
        "path": f"{LANDING_PATH}/node_locations.csv",
        "opts": {"header": "true"},
        "pk_candidate": "location",
    },
    "streets_list": {
        "path": f"{LANDING_PATH}/streets_list.csv",
        "opts": {"header": "true"},
        "pk_candidate": "street_id",
    },
    "streets": {
        "path": f"{LANDING_PATH}/streets.csv",
        "opts": {"header": "true"},
        # No single-column key — grain is the composite (street_id, date).
        # Verified as duplicate-free below in the composite-grain check.
        "pk_candidate": None,
    },
}
 
# Composite-grain candidates: (file_name, [key_columns]) — checked separately
# from the single-column pk_candidate path above.
COMPOSITE_GRAINS = {
    "streets": ["street_id", "date"],
}
 
print(f"INFO: Profiling initialized for landing path: {LANDING_PATH}")

## Date Audit Function

In [0]:
def profile_file(name, path, options, pk_candidate):
    print(f"\n{'='*95}\n DATA AUDIT REPORT: {name.upper()}\n{'='*95}")
 
    df = (spark.read
          .option("inferSchema", False)
          .option("encoding", "UTF-8")
          .options(**options)
          .csv(path))
 
    total_rows = df.count()
    if total_rows == 0:
        print(f" WARNING: File '{name}' is empty.")
        return None, 0
 
    quality_exprs = []
    for c in df.columns:
        quality_exprs += [
            count(when(isnull(col(c)) | (trim(col(c)) == ""), c)).alias(f"{c}_nulls"),
            countDistinct(col(c)).alias(f"{c}_distinct"),
        ]
    audit_results = df.select(quality_exprs).toPandas().transpose()
    audit_results.columns = ["Value"]
 
    profile_rows = []
    for c in df.columns:
        null_count = int(audit_results.loc[f"{c}_nulls", "Value"])
        distinct_count = int(audit_results.loc[f"{c}_distinct", "Value"])
        profile_rows.append({
            "Column_Name": c,
            "Total_Count": total_rows,
            "Null_Count": null_count,
            "Null_Percentage": round((null_count / total_rows) * 100, 2),
            "Distinct_Values": distinct_count,
            "Completeness": f"{round(100 - (null_count / total_rows * 100), 2)}%",
        })
 
    pdf_final = pd.DataFrame(profile_rows)
    print(f" Dataset Stats: {total_rows:,} Rows | {len(df.columns)} Columns")
 
    try:
        display(pdf_final.style.background_gradient(cmap="YlOrRd", subset=["Null_Percentage"]))
    except Exception:
        display(pdf_final)
 
    # Primary-key / duplicate check — only where a real candidate key exists.
    if pk_candidate and pk_candidate in df.columns:
        distinct_rows = df.select(pk_candidate).distinct().count()
        duplicate_count = total_rows - distinct_rows
        integrity_summary = pd.DataFrame([
            {"Metric": "Primary Key Candidate", "Status": pk_candidate},
            {"Metric": "Duplicate Records", "Status": f"{duplicate_count:,}" if duplicate_count > 0 else "0 Duplicates"},
            {"Metric": "Uniqueness Ratio", "Status": f"{round((distinct_rows / total_rows) * 100, 2)}%"},
        ])
    else:
        integrity_summary = pd.DataFrame([
            {"Metric": "Primary Key Candidate", "Status": "None — no unique row-level key in this file (see assumptions doc)"},
        ])
    display(integrity_summary)
 
    return df, total_rows
 


## Automated Audit Execution

In [0]:
profiled = {}
for name, cfg in FILES.items():
    try:
        df, rows = profile_file(name, cfg["path"], cfg["opts"], cfg["pk_candidate"])
        profiled[name] = {"df": df, "rows": rows}
    except Exception as e:
        print(f" ERROR profiling {name}: {str(e)}")
        raise


## Composite Grain Duplicate Checks

In [0]:
for name, key_cols in COMPOSITE_GRAINS.items():
    if name in profiled and profiled[name]["df"] is not None:
        df = profiled[name]["df"]
        total = profiled[name]["rows"]
        distinct_pairs = df.select(*key_cols).distinct().count()
        dup_count = total - distinct_pairs
        print(f"{name}: composite grain {tuple(key_cols)} — "
              f"{distinct_pairs:,} distinct pairs, {dup_count:,} duplicate rows "
              f"({round(dup_count / total * 100, 4)}%)")
        if dup_count > 0:
            print(f"  WARNING: {name} grain is not clean — dedup needed before Silver.")

 


## Data Quality Checks

In [0]:
# Check 1: node_locations — any invalid (0,0) coordinates?
if "node_locations" in profiled and profiled["node_locations"]["df"] is not None:
    bad_coords = profiled["node_locations"]["df"].filter(
        (col("latitude") == "0.0") | (col("longitude") == "0.0")
    )
    bad_count = bad_coords.count()
    print(f"node_locations rows with invalid (0,0) coordinates: {bad_count}")
    if bad_count > 0:
        display(bad_coords)
 
# Check 2: cars — enter/exit distinct-value range (confirms these are counts, not hours)
if "cars" in profiled and profiled["cars"]["df"] is not None:
    from pyspark.sql.functions import max as spark_max
    max_enter = profiled["cars"]["df"].agg(spark_max(col("enter").cast("int"))).collect()[0][0]
    max_exit = profiled["cars"]["df"].agg(spark_max(col("exit").cast("int"))).collect()[0][0]
    print(f"cars.enter max value: {max_enter} | cars.exit max value: {max_exit}")
    if max_enter is not None and max_enter > 23:
        print("  -> Confirms enter/exit are vehicle COUNTS, not hour-of-day values (see assumptions doc).")
 
# Check 3: streets — raining out-of-range values (negative / >100 on what
# looks like a 0-100 percentage-style field)
if "streets" in profiled and profiled["streets"]["df"] is not None:
    from pyspark.sql.functions import min as spark_min
    s_df = profiled["streets"]["df"]
    rain_stats = s_df.agg(
        spark_min(col("raining").cast("double")).alias("min_val"),
        spark_max(col("raining").cast("double")).alias("max_val"),
    ).collect()[0]
    neg_count = s_df.filter(col("raining").cast("double") < 0).count()
    total_streets_rows = profiled["streets"]["rows"]
    print(f"streets.raining range: min={rain_stats['min_val']}, max={rain_stats['max_val']}")
    print(f"streets.raining negative rows: {neg_count:,} "
          f"({round(neg_count / total_streets_rows * 100, 4)}%)")
    if rain_stats["min_val"] is not None and rain_stats["min_val"] < 0:
        print("  -> WARNING: raining has values below 0 on what reads as a 0-100 scale. "
              "Flagged as a probable sentinel/measurement-floor artifact, not real negative "
              "rainfall — see assumptions doc. Needs a Silver-layer decision (clip vs quarantine).")
 
# Check 4: streets — light cardinality sanity check (near-unique per row)
if "streets" in profiled and profiled["streets"]["df"] is not None:
    light_distinct = profiled["streets"]["df"].select(countDistinct(col("light"))).collect()[0][0]
    total_streets_rows = profiled["streets"]["rows"]
    ratio = round(light_distinct / total_streets_rows * 100, 4)
    print(f"streets.light distinct values: {light_distinct:,} / {total_streets_rows:,} rows ({ratio}%)")
    if ratio > 99.9:
        print("  -> light is effectively unique per row — plausible for a continuous sensor "
              "reading (lux), but confirm it isn't a mis-mapped ID/timestamp column before Silver.")
 
print(f"\n{'='*55}")
print(f"  PROFILING COMPLETE")
print(f"{'='*55}")
for name, cfg in profiled.items():
    print(f"  {name:<16} {cfg['rows']:>12,} rows")
print(f"{'='*55}")